# Generate Word documents of results for readability

In [22]:
# Overview of JSON structure for prompting without detailing  entries
import json

def show_structure_limited(file_path, file_type='json', max_entries=2, max_value_length=10):
    """
    Display structure of JSON files with limited entries and truncated values
    """
    print(f"\n=== Structure of {file_path} ===")
    
    try:
        if file_type == 'jsonl':
            # Handle JSONL format
            entries = []
            with open(file_path, 'r', encoding='utf-8') as f:
                for i, line in enumerate(f):
                    if i >= max_entries:
                        break
                    entries.append(json.loads(line))
        else:
            # Handle regular JSON format
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    entries = data[:max_entries]
                elif isinstance(data, dict):
                    # For dict, take first N key-value pairs
                    entries = [dict(list(data.items())[:max_entries])]
                else:
                    entries = [data]
        
        # Display structure with truncated values
        for i, entry in enumerate(entries):
            print(f"\nEntry {i+1}:")
            display_dict_structure(entry, max_value_length, indent=2)
            
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

def display_dict_structure(obj, max_length, indent=0):
    """
    Recursively display dictionary structure with truncated values
    """
    prefix = " " * indent
    
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, (dict, list)):
                print(f"{prefix}{key}: {type(value).__name__}")
                if isinstance(value, dict) and len(value) > 0:
                    # Show first few keys of nested dict
                    first_keys = list(value.keys())
                    if len(first_keys) < len(value):
                        first_keys.append('...')
                    print(f"{prefix}  keys: {first_keys}")
                elif isinstance(value, list) and len(value) > 0:
                    print(f"{prefix}  length: {len(value)}")
                    print(f"{prefix}  first_item_type: {type(value[0]).__name__}")
                    # Show first item structure if it's a dict
                    if isinstance(value[0], dict):
                        first_keys = list(value[0].keys())[:3]
                        if len(first_keys) < len(value[0]):
                            first_keys.append('...')
                        print(f"{prefix}  first_item_keys: {first_keys}")
                    # Show key-value pairs for first few items
                    for i, item in enumerate(value[:2]):  # Show first 2 items
                        if isinstance(item, dict):
                            print(f"{prefix}  [{i}]: dict with keys {list(item.keys())[:3]}")
                            for k, v in list(item.items()):  # First 2 key-value pairs
                                if isinstance(v, str):
                                    truncated = v[:max_length] + "..." if len(v) > max_length else v
                                    print(f"{prefix}    {k}: '{truncated}'")
                                else:
                                    print(f"{prefix}    {k}: {v}")
                        else:
                            print(f"{prefix}  [{i}]: {item}")
            else:
                # Truncate string values
                if isinstance(value, str):
                    truncated = value[:max_length] + "..." if len(value) > max_length else value
                    print(f"{prefix}{key}: '{truncated}'")
                else:
                    print(f"{prefix}{key}: {value}")
    elif isinstance(obj, list):
        print(f"{prefix}List with {len(obj)} items")
        if len(obj) > 0:
            print(f"{prefix}First item type: {type(obj[0]).__name__}")
            # Show key-value pairs for first few items if they are dicts
            for i, item in enumerate(obj[:2]):  # Show first 2 items
                if isinstance(item, dict):
                    print(f"{prefix}[{i}]: dict with keys {list(item.keys())[:3]}")
                    for k, v in list(item.items()):  # First 2 key-value pairs
                        if isinstance(v, str):
                            truncated = v[:max_length] + "..." if len(v) > max_length else v
                            print(f"{prefix}  {k}: '{truncated}'")
                        else:
                            print(f"{prefix}  {k}: {v}")
                else:
                    print(f"{prefix}[{i}]: {item}")

# Show structure of all 3 files
corpus_path = f'{input_dir}/corpus.jsonl'
retrieved_path = f'{input_dir}/retrieved_trials.json'
matching_path = '/work/sander_git/TrialGPT/results/matching_results_mstro_ollama:gemma3:27b.json'

show_structure_limited(corpus_path, 'jsonl')
show_structure_limited(retrieved_path, 'json')
show_structure_limited(matching_path, 'json')


=== Structure of /work/sander_git/TrialGPT/dataset/mstro//corpus.jsonl ===

Entry 1:
  _id: 'Ependymoom...'
  title: 'Ependymoom...'
  text: 'Study Type...'
  metadata: dict
    keys: ['brief_title', 'phase', 'drugs', 'drugs_list', 'diseases', 'diseases_list', 'enrollment', 'inclusion_criteria', 'exclusion_criteria', 'brief_summary']

Entry 2:
  _id: 'ERROR'
  title: 'ERROR'
  text: 'Study Type...'
  metadata: dict
    keys: ['brief_title', 'phase', 'drugs', 'drugs_list', 'diseases', 'diseases_list', 'enrollment', 'inclusion_criteria', 'exclusion_criteria', 'brief_summary']

=== Structure of /work/sander_git/TrialGPT/dataset/mstro//retrieved_trials.json ===

Entry 1:
  patient_id: '006_MDO_C_...'
  patient: '& zuyderla...'
  0: list
    length: 1
    first_item_type: dict
    first_item_keys: ['NCTID', 'title', 'text', '...']
    [0]: dict with keys ['NCTID', 'title', 'text']
      NCTID: 'Biobank 00...'
      title: 'Biobank 00...'
      text: 'Study Type...'
      brief_title: 'Biob

In [23]:
!pip install docx

/bin/bash: line 1: /work/sander_git/TrialGPT/.venv/bin/pip: cannot execute: required file not found


In [ ]:
import json
import os
from docx import Document
from docx.shared import RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH


input_dir = '/work/sander_git/TrialGPT/dataset/mstro/'


def generate_patient_reports(
    corpus_path=f'{input_dir}/corpus.jsonl',
    retrieved_path=f'{input_dir}/retrieved_trials.json',
    matching_path='/work/sander_git/TrialGPT/results/matching_results_mstro_ollama:gemma3:27b.json',
    output_dir=f'{input_dir}/patient_reports'
):
    """
    Generate a Word document report for each patient based on trial matching results,
    with explicit criteria text, trial summary, and colored status for inclusion/exclusion.
    Patient information is assumed to be OCR output from a PDF.
    """
    # Load corpus for full trial definitions
    corpus = {}
    with open(corpus_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            corpus[item['_id']] = item

    # Load retrieved trials mapping patient_id to trial entries
    with open(retrieved_path, 'r', encoding='utf-8') as f:
        retrieved = json.load(f)
    # Note: patient_info is OCR output from PDF
    patient_info = {entry['patient_id']: entry.get('patient', '').strip() for entry in retrieved}

    # Load matching results
    with open(matching_path, 'r', encoding='utf-8') as f:
        matching = json.load(f)

    os.makedirs(output_dir, exist_ok=True)

    # Status to color mapping
    status_colors = {
        # Inclusion labels
        "included": RGBColor(0x00, 0x80, 0x00),       # dark green (meets inclusion)
        "not included": RGBColor(0xC0, 0x00, 0x00),   # dark red   (does not meet inclusion)

        # Exclusion labels
        "excluded": RGBColor(0xC0, 0x00, 0x00),        # dark red   (meets exclusion → exclude)
        "not excluded": RGBColor(0x00, 0x80, 0x00),    # dark green (does not meet exclusion → okay)

        # Shared “uncertain” or “N/A” labels
        "not enough information": RGBColor(0x80, 0x80, 0x80),  # gray
        "not applicable": RGBColor(0x80, 0x80, 0x80)           # gray
    }

    for patient_id, pages_dict in matching.items():
        doc = Document()
        doc.add_heading(f'Report for Patient: {patient_id}', level=1)

        # Patient Information (OCR output)
        doc.add_heading('Patient Information (OCR from PDF)', level=2)
        doc.add_paragraph(patient_info.get(patient_id, 'No OCR patient info available.'))

        doc.add_heading('Matched Trials Overview', level=2)
        any_trial = False

        for page_num, trial_matches in pages_dict.items():
            if not trial_matches:
                continue

            for trial_id, result in trial_matches.items():
                any_trial = True
                # Fetch full trial entry for criteria and summary
                trial = corpus.get(trial_id, {})

                # Access metadata fields
                metadata = trial.get('metadata', {})
                summary_text = metadata.get('brief_summary', 'No trial summary available.')
                inc_text = metadata.get('inclusion_criteria', 'No inclusion criteria text available.')
                exc_text = metadata.get('exclusion_criteria', 'No exclusion criteria text available.')

                doc.add_heading(f'Trial: {trial_id}', level=3)

                # Add Trial Summary
                doc.add_heading('Trial Summary', level=4)
                doc.add_paragraph(summary_text)

                # Inclusion criteria full text
                doc.add_heading('Inclusion Criteria (Full Text)', level=4)
                doc.add_paragraph(inc_text)

                # Inclusion evaluation table
                doc.add_heading('Inclusion Evaluation', level=4)
                inclusion_dict = result.get('inclusion', {})
                if not inclusion_dict:
                    doc.add_paragraph('No inclusion criteria evaluated.')
                else:
                    # Create table with two columns: Statement | Status
                    table = doc.add_table(rows=1, cols=2)
                    table.autofit = False

                    # Compute column widths: status column = 20% of available width
                    section = doc.sections[0]
                    available_width = section.page_width - section.left_margin - section.right_margin  # Twips
                    status_col_width = int(available_width * 0.20)
                    stmt_col_width = available_width - status_col_width

                    # Set header row text and alignment
                    hdr_cells = table.rows[0].cells
                    hdr_cells[0].text = 'Statement'
                    hdr_cells[1].text = 'Status'
                    for cell in hdr_cells:
                        for paragraph in cell.paragraphs:
                            paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

                    # Set widths for header cells
                    hdr_cells[0].width = stmt_col_width
                    hdr_cells[1].width = status_col_width

                    # Populate rows
                    for _, (statement, sentences, status) in inclusion_dict.items():
                        row_cells = table.add_row().cells
                        # Statement column
                        stmt_para = row_cells[0].paragraphs[0]
                        stmt_para.text = statement
                        # Status column with colored text
                        status_para = row_cells[1].paragraphs[0]
                        status_run = status_para.add_run(status)
                        status_color = status_colors.get(status, RGBColor(0, 0, 0))
                        status_run.font.color.rgb = status_color
                        # Set widths for each new row's cells
                        row_cells[0].width = stmt_col_width
                        row_cells[1].width = status_col_width

                # Exclusion criteria full text
                doc.add_heading('Exclusion Criteria (Full Text)', level=4)
                doc.add_paragraph(exc_text)

                # Exclusion evaluation table
                doc.add_heading('Exclusion Evaluation', level=4)
                exclusion_dict = result.get('exclusion', {})
                if not exclusion_dict:
                    doc.add_paragraph('No exclusion criteria evaluated.')
                else:
                    # Create table with two columns: Statement | Status
                    table = doc.add_table(rows=1, cols=2)
                    table.autofit = False

                    # Reuse computed column widths
                    section = doc.sections[0]
                    available_width = section.page_width - section.left_margin - section.right_margin
                    status_col_width = int(available_width * 0.20)
                    stmt_col_width = available_width - status_col_width

                    # Set header row text and alignment
                    hdr_cells = table.rows[0].cells
                    hdr_cells[0].text = 'Statement'
                    hdr_cells[1].text = 'Status'
                    for cell in hdr_cells:
                        for paragraph in cell.paragraphs:
                            paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

                    # Set widths for header cells
                    hdr_cells[0].width = stmt_col_width
                    hdr_cells[1].width = status_col_width

                    # Populate rows
                    for _, (statement, sentences, status) in exclusion_dict.items():
                        row_cells = table.add_row().cells
                        # Statement column
                        stmt_para = row_cells[0].paragraphs[0]
                        stmt_para.text = statement
                        # Status column with colored text
                        status_para = row_cells[1].paragraphs[0]
                        status_run = status_para.add_run(status)
                        status_color = status_colors.get(status, RGBColor(0, 0, 0))
                        status_run.font.color.rgb = status_color
                        # Set widths for each new row's cells
                        row_cells[0].width = stmt_col_width
                        row_cells[1].width = status_col_width

        if not any_trial:
            doc.add_paragraph('No trials matched for this patient.')

        output_path = os.path.join(output_dir, f'{patient_id}.docx')
        doc.save(output_path)
        print(f"Generated report: {output_path}")


# Run the function:
generate_patient_reports()
